In [16]:
import json
import os
from datetime import datetime, timedelta
from crewai import Agent, Task, Crew, Process
from crewai_tools import CSVSearchTool

In [2]:
from dotenv import load_dotenv
_ = load_dotenv()

import os 
API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo-0125", api_key=API_KEY)

In [4]:
csv_carteira = CSVSearchTool(csv="ativos.csv")

Agente 1 - Gerente do cliente

In [5]:
gerente_cliente = Agent(
    role="Gerente de Carteira do Cliente",
    goal="Obtenha a pergunta do cliente sobre o ativo {ticket} e pesquise as ações no arquivo CSV da carteira do cliente",
    backstory="""
    Você é o gerente de clientes da carteira de investimentos do cliente.
    Você é o primeiro contato do cliente e fornece as informações para as demais análises com o ticket do ativo e informações de carteira necessárias
    """,
    verbose=True,
    max_iter=5,
    tools=[csv_carteira],
    memory=True
)

In [6]:
obter_carteira_cliente = Task(
    description=""",
    Use a pergunta do cliente e encontre o ativo {ticket} no arquivo CSV.
    Forneça se o ativo está na carteira do cliente e se estiver, forneça o preço médio que ele pagou e o número total de ações em posse.
    """,
    expected_output="Se o cliente possuir os ativos, forneça o preço médio e o total de ações dos ativos",
    agent=gerente_cliente
)

In [7]:
import yfinance as yf

def pega_preco_ativo(ticket):
    data_final = datetime.today()
    data_inicial = data_final - timedelta(days=365)
    ativo = yf.download(ticket, start=data_inicial.strftime('%Y-%m-%d'), end=data_final.strftime('%Y-%m-%d'))
    return ativo

In [8]:
resultado = pega_preco_ativo("PETR4.SA")
resultado

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,PETR4.SA,PETR4.SA,PETR4.SA,PETR4.SA,PETR4.SA
Date,,,,,
2025-05-06,27.378172,27.596107,27.205639,27.341850,52751300
2025-05-07,27.505302,27.514382,27.151156,27.514382,35050400
2025-05-08,27.886688,28.177271,27.650593,27.777721,44665000
2025-05-09,28.068302,28.313480,27.823124,28.240835,25075600
2025-05-12,28.740271,29.212463,28.740271,28.967287,53493600
...,...,...,...,...,...
2026-04-28,47.520000,48.040001,47.459999,47.650002,29389700
2026-04-29,48.959999,49.299999,48.000000,48.099998,47685000


Agente 2 - Analista de Ações

In [9]:
analista_acoes = Agent(
    role="Analista senior de preço de ações",
    goal="Encontre o preço da ação {ticket} e analise suas tendências para fornecer uma recomendação de compra, venda ou manutenção para o gerente de carteira, além de comparar com o preço que foi pago",
    backstory="""
    Você é um analista de  muito experiente.
    Você é responsável por analisar os ativos e fornecer informações sobre o preço do ativo para o gerente de  e fazer previsões sobre seu preço futuro.
    """,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [10]:
from crewai.tools import BaseTool
from pydantic import Field

In [11]:
class YahooFinanceTool(BaseTool):
    name: str = "Yahoo Finance Tool"
    description: str = "Use esta ferramenta para obter informações sobre o preço de um ativo, no último ano, usando a biblioteca yfinance. Forneça o ticket do ativo para obter as informações necessárias."

    def _run(self, ticket: str):
        """Executa a busca de preços de ações para o ativo fornecido usando a biblioteca yfinance."""
        try:
            data_final = datetime.today()
            data_inicial = data_final - timedelta(days=365)
            ativo = yf.download(ticket, start=data_inicial.strftime('%Y-%m-%d'), end=data_final.strftime('%Y-%m-%d'))
            return ativo.to_dict()
        except Exception as e:
            return f"Erro ao obter dados do Yahoo Finance: {str(e)}"

In [12]:
yfinance_tool = YahooFinanceTool()
type(yfinance_tool)

__main__.YahooFinanceTool

In [13]:
obter_preco_acao = Task(
    description="""
    Use a ferramenta Yahoo Finance Tool para obter o preço da ação {ticket} e analisar suas tendências para fornecer uma recomendação de compra, venda ou manutenção para o gerente de carteira, além de comparar com o preço que foi pago.
    """,
    expected_output="Forneça uma recomendação de compra, venda ou manutenção para o gerente de carteira, além de comparar com o preço que foi pago e especifique uma tendencia atual do preço da acao tanto para cima quanto para baixo",
    agent=analista_acoes,
    tools=[yfinance_tool]
)

Agente 3 - Analista de notícias

In [14]:
analista_noticias = Agent(
    role="Analista senior de notícias de ações",
    goal="Encontre as últimas notícias sobre o ativo {ticket} e analise seu impacto potencial no preço da ação para fornecer uma recomendação de compra, venda ou manutenção para o gerente de carteira",
    backstory="""
    Você é um analista muito experiente.
    Você é responsável por analisar as notícias relacionadas aos ativos e fornecer informações sobre o impacto potencial dessas notícias no preço do ativo para o gerente de carteira.
    """,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [19]:
from langchain_community.tools import DuckDuckGoSearchResults
searchTool = DuckDuckGoSearchResults(backend="news", num_results=10)

In [25]:
obter_noticias = Task(
    description=f"""
    Use a ferramenta DuckDuckGo News Tool para obter as últimas notícias sobre o ativo e analisar seu impacto potencial no preço da ação para fornecer uma recomendação de compra, venda ou manutenção para o gerente de carteira.
    A data atual é {datetime.now()}
    Componha os resultados em um relatório útil
    """,
    expected_output="Forneça uma recomendação de compra, venda ou manutenção para o gerente de carteira, além de analisar o impacto potencial das notícias no preço do ativo.",
    agent=analista_noticias,
    tool=[searchTool]
)

Agente 4 - Analista chefe de ações

In [26]:
analista_chefe = Agent(
    role="Analista chefe de investimentos",
    goal="Com base nas análises do analista de ações e do analista de notícias, forneça uma recomendação final de compra, venda ou manutenção para o gerente de carteira, considerando tanto as tendências de preço quanto o impacto das notícias no ativo {ticket}.",
    backstory="""
    Você é um analista chefe de investimentos altamente experiente.
    Você é responsável por revisar as análises fornecidas pelos analistas de ações e notícias, e fornecer uma recomendação final para o gerente de carteira com base em todas as informações disponíveis.
    """,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [27]:
recomendar_acao = Task(
    description="""
    Com base nas análises do analista de ações e do analista de notícias, forneça uma recomendação final de compra, venda ou manutenção para o gerente de carteira, considerando tanto as tendências de preço quanto o impacto das notícias no ativo {ticket}.
    Se os relatórios não forem conclusivos, pode solicitar mais análises ou informações adicionais para chegar a uma recomendação mais informada.
    """,
    expected_output="Forneça uma recomendação final de compra, venda ou manutenção para o gerente de carteira, considerando tanto as tendências de preço quanto o impacto das notícias no ativo.",
    agent=analista_chefe,
    context=[obter_carteira_cliente, obter_preco_acao, obter_noticias]
)